# Gradient Sports API — Exemplos de Uso

Importa o cliente de `gradient_client.py`.  
A autenticação é lida automaticamente do arquivo `.env` (`BEARER_TOKEN`).

In [15]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [16]:
from gradient_client import GradientSportsClient

## Usage examples

In [17]:
client = GradientSportsClient()

# health check
client.get_status()

{'data': {'status': 'ok'}}

In [18]:
# competitions & seasons the account can access
df_competitions = client.get_competitions(as_dataframe=True)
df_competitions

,datasets,season,competition.id,competition.name
0,physical_metrics,2020-2021,1,Premier League
1,events,2020-2021,1,Premier League
2,sprints,2020-2021,1,Premier League
3,high_speed_runs,2020-2021,1,Premier League
4,physical_metrics,2021-2022,1,Premier League
5,events,2021-2022,1,Premier League
6,sprints,2021-2022,1,Premier League
7,high_speed_runs,2021-2022,1,Premier League
8,physical_metrics,2022-2023,1,Premier League
9,events,2022-2023,1,Premier League


In [19]:
# teams the account can access
df_teams = client.get_teams(as_dataframe=True)
df_teams.tail()

,team.id,team.name,competition.id,competition.name,dataset
225,15.0,Sheffield United,1,Premier League,high_speed_runs
226,425.0,América Mineiro,42,Brasileiro Série A,physical_metrics
227,425.0,América Mineiro,42,Brasileiro Série A,events
228,425.0,América Mineiro,42,Brasileiro Série A,sprints
229,425.0,América Mineiro,42,Brasileiro Série A,high_speed_runs


In [20]:
# all games — or filter by season / competition / team
# as_dataframe=True → one row per game, nested dicts dot-expanded
games = client.get_games(season="2021-2022", competition_id=1, as_dataframe=True)
games.head()

,id,date,season,teamExtraTimeStartSide,teamStartSide,venueType,team.id,team.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name,stadium.name,stadium.length,stadium.width
0,1195,2021-11-27,2021-2022,Right,Left,TEAM_HOME,2,Arsenal,1,Premier League,13,Newcastle United,Emirates Stadium,105.0,68.0
1,1139,2021-10-02,2021-2022,Right,Left,TEAM_HOME,55,Leeds United,1,Premier League,18,Watford,Elland Road,105.0,68.0
2,1277,2022-01-02,2021-2022,Right,Right,TEAM_HOME,6,Chelsea,1,Premier League,10,Liverpool,Stamford Bridge,103.0,67.5
3,1199,2021-11-28,2021-2022,Right,Right,TEAM_HOME,6,Chelsea,1,Premier League,12,Manchester United,Stamford Bridge,103.0,67.5
4,1422,2022-04-30,2021-2022,Left,Right,OPPONENT_HOME,5,Burnley,1,Premier League,18,Watford,Vicarage Road,105.0,68.0


In [21]:
# pick a game id from the list above
GAME_ID = games["id"].iloc[0]

# structured events — one row per possession event
df_game_events = client.get_game_events(GAME_ID, as_dataframe=True)
df_game_events = df_game_events.sort_values(by="poss.startGameClock")
df_game_events.head()

,id,competitionId,gameId,season,period,periodDescription,startGameClock,startFormattedGameClock,homeTeam,gameEventType,gameEventTypeDescription,setpieceType,setpieceTypeDescription,touches,touchesInBox,team.id,team.name,player.id,player.name,poss.id,poss.type,poss.typeDescription,poss.startGameClock,poss.endGameClock,poss.period,poss.nonEvent,poss.ballHeightType,poss.highPointType,poss.bodyType,poss.player.id,poss.player.name,poss.team.id,poss.team.name
0,2180404,None,None,None,1,First half,0,00:00,True,FIRSTKICKOFF,First half kick off,K,Kickoff,1.0,0,2.0,Arsenal,1688.0,Martin Ödegaard,2055749,PA,Pass,0,65.899,1,False,G,G,R,1688.0,Martin Ödegaard,2.0,Arsenal
1,2180412,None,None,None,1,First half,1,00:01,True,OTB,A possession with a player on the ball,O,Open Play,2.0,0,2.0,Arsenal,281.0,Aaron Ramsdale,2055757,PA,Pass,1,69.269,1,False,G,A,R,281.0,Aaron Ramsdale,2.0,Arsenal
2,2180418,None,None,None,1,First half,7,00:07,True,OTB,A possession with a player on the ball,O,Open Play,1.0,0,2.0,Arsenal,2087.0,Takehiro Tomiyasu,2055763,CH,Challenge,7,72.973,1,False,NaN,NaN,NaN,2087.0,Takehiro Tomiyasu,2.0,Arsenal
3,2180418,None,None,None,1,First half,7,00:07,True,OTB,A possession with a player on the ball,O,Open Play,1.0,0,2.0,Arsenal,2087.0,Takehiro Tomiyasu,2055764,PA,Pass,7,72.973,1,False,A,A,HE,2087.0,Takehiro Tomiyasu,2.0,Arsenal
4,2180424,None,None,None,1,First half,13,00:13,False,OTB,A possession with a player on the ball,T,Throw in,1.0,0,13.0,Newcastle United,326.0,Matt Ritchie,2055769,PA,Pass,13,79.580,1,False,A,A,2H,326.0,Matt Ritchie,13.0,Newcastle United


In [22]:
df_game_events.columns

Index(['id', 'competitionId', 'gameId', 'season', 'period',
       'periodDescription', 'startGameClock', 'startFormattedGameClock',
       'homeTeam', 'gameEventType', 'gameEventTypeDescription', 'setpieceType',
       'setpieceTypeDescription', 'touches', 'touchesInBox', 'team.id',
       'team.name', 'player.id', 'player.name', 'poss.id', 'poss.type',
       'poss.typeDescription', 'poss.startGameClock', 'poss.endGameClock',
       'poss.period', 'poss.nonEvent', 'poss.ballHeightType',
       'poss.highPointType', 'poss.bodyType', 'poss.player.id',
       'poss.player.name', 'poss.team.id', 'poss.team.name'],
      dtype='str')

In [23]:
# physical metrics for a single game — one row per (player × metric)
df_physical_metrics = client.query_game_physical_metrics(GAME_ID, possessions=["ALL"], as_dataframe=True).head()
df_physical_metrics.tail()

,location,season,possession,gameDate,playerPosition,player.id,player.name,team.id,team.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name,metric.name,metric.p90,metric.raw
0,Away,2021-2022,ALL,2021-11-27,RCB,320,Fabian Schär,13,Newcastle United,1,Premier League,2,Arsenal,accelerations,65.99,73.0
1,Away,2021-2022,ALL,2021-11-27,RCB,320,Fabian Schär,13,Newcastle United,1,Premier League,2,Arsenal,athleticism_score,NaN,9.9
2,Away,2021-2022,ALL,2021-11-27,RCB,320,Fabian Schär,13,Newcastle United,1,Premier League,2,Arsenal,decelerations,64.18,71.0
3,Away,2021-2022,ALL,2021-11-27,RCB,320,Fabian Schär,13,Newcastle United,1,Premier League,2,Arsenal,game_appearances,NaN,1.0
4,Away,2021-2022,ALL,2021-11-27,RCB,320,Fabian Schär,13,Newcastle United,1,Premier League,2,Arsenal,game_starts,NaN,1.0


In [24]:
# sprints — one row per sprint
df_sprints = client.get_game_sprints(GAME_ID, as_dataframe=True)
df_sprints.head()

,id,position,started,period,season,distance,videoUrl,gameDate,gameId,periodElapsedTimeEnd,periodElapsedTimeStart,periodGameClockTimeEnd,periodGameClockTimeStart,runTime,shirtNumber,speedKmh,videoEndAt,videoStartAt,xEnd,xStart,yEnd,yStart,player.id,player.name,team.id,team.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name
0,787d70bab925562d94a9dd7085d980eb,RB,True,1,2021-2022,7.102868,https://epitome.gradientsports.com/film_room/c...,2021-11-27,1195,19.019252,18.018251,19.019252,18.018251,1.001001,18,27.063712,84918.252,83917.251,-20.366792,-13.402675,-24.129912,-25.526983,2087,Takehiro Tomiyasu,2,Arsenal,1,Premier League,13,Newcastle United
1,d945263f5284d8a4f3d37ba0390fb330,RCB,True,1,2021-2022,7.269062,https://epitome.gradientsports.com/film_room/c...,2021-11-27,1195,19.019252,18.018251,19.019252,18.018251,1.001001,4,28.506783,84918.252,83917.251,-23.946144,-16.920189,-27.149198,-29.013396,1916,Ben White,2,Arsenal,1,Premier League,13,Newcastle United
2,d9ca5892a01f523da17796fd024fa89d,CM,True,1,2021-2022,14.454137,https://epitome.gradientsports.com/film_room/c...,2021-11-27,1195,20.020253,18.018251,20.020253,18.018251,2.002002,5,26.104924,85919.253,83917.251,-30.558865,-16.700667,-18.402495,-22.483377,1380,Thomas Partey,2,Arsenal,1,Premier League,13,Newcastle United
3,e14195a9de8f48ce51d39701cfafb8fc,RCB,True,1,2021-2022,6.857521,https://epitome.gradientsports.com/film_room/c...,2021-11-27,1195,21.021254,20.020253,21.021254,20.020253,1.001001,4,25.760577,86920.254,85919.253,-38.070105,-31.302563,-23.740103,-24.847343,1916,Ben White,2,Arsenal,1,Premier League,13,Newcastle United
4,f2d517257a20912b431061606a3f4d43,RM,True,1,2021-2022,7.165298,https://epitome.gradientsports.com/film_room/c...,2021-11-27,1195,299.299532,298.298531,299.299532,298.298531,1.001001,7,28.178062,365198.532,364197.531,26.925973,21.253687,-13.999594,-18.377570,163,Bukayo Saka,2,Arsenal,1,Premier League,13,Newcastle United


In [25]:
# high speed runs — one row per run
df_high_speed_runs = client.get_game_high_speed_runs(GAME_ID, as_dataframe=True)
df_high_speed_runs.head()

,id,position,started,period,season,distance,videoUrl,gameDate,gameId,periodElapsedTimeEnd,periodElapsedTimeStart,periodGameClockTimeEnd,periodGameClockTimeStart,runTime,shirtNumber,speedKmh,videoEndAt,videoStartAt,xEnd,xStart,yEnd,yStart,player.id,player.name,team.id,team.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name
0,61f1f2ffcdfe109749ab6b5701e5c982,CM,True,1,2021-2022,11.170186,https://epitome.gradientsports.com/film_room/c...,2021-11-27,1195,6.006239,4.004237,6.006239,4.004237,2.002002,5,20.564753,71905.239,69903.237,-0.396770,-5.261235,-16.714116,-6.806077,1380,Thomas Partey,2,Arsenal,1,Premier League,13,Newcastle United
1,3a66fd84651f2351a45f05009879da45,LB,True,1,2021-2022,17.089148,https://epitome.gradientsports.com/film_room/c...,2021-11-27,1195,20.020253,17.017250,20.020253,17.017250,3.003003,20,21.078848,85919.253,82916.250,-33.881123,-17.140044,-4.937409,-6.546982,5044,Nuno Tavares,2,Arsenal,1,Premier League,13,Newcastle United
2,9872aa402b51500554362730ed36ecfd,CM,True,1,2021-2022,5.726664,https://epitome.gradientsports.com/film_room/c...,2021-11-27,1195,18.018251,17.017250,18.018251,17.017250,1.001001,5,23.422105,83917.251,82916.250,-16.700667,-11.180960,-22.483377,-24.008996,1380,Thomas Partey,2,Arsenal,1,Premier League,13,Newcastle United
3,c235b81f3c6d3ed3c9d32d2e39736b85,LCB,True,1,2021-2022,19.507980,https://epitome.gradientsports.com/film_room/c...,2021-11-27,1195,20.020253,17.017250,20.020253,17.017250,3.003003,6,23.373410,85919.253,82916.250,-33.289651,-17.018455,-9.231419,-19.826326,1894,Gabriel Magalhães,2,Arsenal,1,Premier League,13,Newcastle United
4,a1d2d5dce22eb48df5dbc23a968f6a33,RB,True,1,2021-2022,5.755992,https://epitome.gradientsports.com/film_room/c...,2021-11-27,1195,18.018251,17.017250,18.018251,17.017250,1.001001,18,23.574711,83917.251,82916.250,-13.402675,-7.721781,-25.526983,-26.453744,2087,Takehiro Tomiyasu,2,Arsenal,1,Premier League,13,Newcastle United


In [26]:
# cross-player physical metrics — one row per (player × metric)
# Supported filter operators:
#   gt / lt          → {"operator": "gt",             "subject": "...", "value": <number>}
#   values_between   → {"operator": "values_between", "subject": "...", "values": [min, max]}
#   above_median     → NOT usable standalone; the API schema requires value/values for every operator
df_physical_metrics = client.query_physical_metrics(
        season="2024-2025",
        competition_ids=[1],
        possession="ALL",
        filters={
            "and": [
                {"operator": "gt", "subject": "game_appearances", "value": 10},
                {"operator": "gt", "subject": "sprints",          "value": 5},
            ]
        },
        as_dataframe=True,
    )
df_physical_metrics

,id,position,dob,firstName,lastName,playedHistory,team.id,team.name,metric.name,metric.p90,metric.p90Percentile,metric.raw,metric.rawPercentile
0,1,CF,1993-07-28,Harry,Kane,"[{'competition_id': 1, 'competition_name': 'Pr...",160,FC Bayern München,accelerations,40.2002,0.333086,4704.000,0.918398
1,1,CF,1993-07-28,Harry,Kane,"[{'competition_id': 1, 'competition_name': 'Pr...",160,FC Bayern München,athleticism_score,NaN,NaN,46.700,0.430267
2,1,CF,1993-07-28,Harry,Kane,"[{'competition_id': 1, 'competition_name': 'Pr...",160,FC Bayern München,decelerations,42.2512,0.278932,4944.000,0.913205
3,1,CF,1993-07-28,Harry,Kane,"[{'competition_id': 1, 'competition_name': 'Pr...",160,FC Bayern München,game_appearances,NaN,NaN,110.000,0.878338
4,1,CF,1993-07-28,Harry,Kane,"[{'competition_id': 1, 'competition_name': 'Pr...",160,FC Bayern München,game_starts,NaN,NaN,109.000,0.930267
...,...,...,...,...,...,...,...,...,...,...,...,...,...
23635,45549,LW,2008-08-29,Rio,Ngumoha,"[{'competition_id': 1, 'competition_name': 'Pr...",10,Liverpool,total_distance,9.1543,0.365727,32.100,0.246291
23636,45549,LW,2008-08-29,Rio,Ngumoha,"[{'competition_id': 1, 'competition_name': 'Pr...",10,Liverpool,total_minutes,NaN,NaN,315.590,0.244065
23637,45549,LW,2008-08-29,Rio,Ngumoha,"[{'competition_id': 1, 'competition_name': 'Pr...",10,Liverpool,visibility_percentage,NaN,NaN,0.401,0.545252
23638,45549,LW,2008-08-29,Rio,Ngumoha,"[{'competition_id': 1, 'competition_name': 'Pr...",10,Liverpool,walk_distance,3.1855,0.727003,11.170,0.244065


In [27]:
# player-level metrics — one row per (game × metric)
players = client.get_players(as_dataframe=True)
PLAYER_ID = players["id"].iloc[-1]

df_player_physical_metrics = client.query_player_physical_metrics(
        PLAYER_ID,
        seasons=["2024-2025"],
        competition_ids=[1],
        possessions=["ALL"],
        as_dataframe=True,
    ).head()

df_player_physical_metrics.tail()

""
